<a href="https://colab.research.google.com/github/Gunnny/dream-atlas/blob/main/notebooks/01_explore_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
!pip install datasets

In [45]:
from datasets import load_dataset
ds = load_dataset("DReAMy-lib/DreamBank-dreams-en")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['series', 'description', 'dreams', 'gender', 'year'],
        num_rows: 22415
    })
})


In [46]:
import pandas as pd

df = ds["train"].to_pandas()
df["word_count"] = df["dreams"].str.split().str.len()

df_clean = df[df["word_count"] >= 5].copy()
print(len(df), "→", len(df_clean))

print("vorher:", len(df_clean))
df_clean = df_clean.drop_duplicates(subset="dreams").reset_index(drop=True)
print("nachher:", len(df_clean))

df_clean.to_parquet("dreams_clean.parquet")

22415 → 22347
vorher: 22347
nachher: 22096


Nur Berichte mit unter 5 Wörtern entfernt (68 Zeilen, 0,3 %) – das sind Titel ohne Text. Kurze Berichte ab 5 Wörtern sind inhaltlich vollständig und bleiben erhalten; gefiltert wird später pro Anwendungsfall über word_count.

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", min_df=2)
X = vectorizer.fit_transform(df_clean["dreams"])
print(X.shape)

zellen = X.shape[0] * X.shape[1]
print("Zellen insgesamt:", zellen)
print("davon besetzt:   ", X.nnz)
print("Füllgrad:        ", round(100 * X.nnz / zellen, 3), "%")
print("Speicher:        ", round(X.data.nbytes / 1e6, 1), "MB")
print("wäre dicht:      ", round(zellen * 8 / 1e9, 2), "GB")

(22096, 24011)
Zellen insgesamt: 530547056
davon besetzt:    1170708
Füllgrad:         0.221 %
Speicher:         9.4 MB
wäre dicht:       4.24 GB


In [48]:
import numpy as np

def search(query, k=5):
    q = vectorizer.transform([query])
    sims = (X @ q.T).toarray().ravel()
    top = np.argsort(-sims)[:k]

    for i in top:
        print(round(float(sims[i]), 3), "|", df_clean["dreams"].iloc[i][:200])
        print("---")

search("I was being chased through a dark house")
search("I was being pursued through a dark house")
search("Ich wurde durch ein dunkles Haus verfolgt")

0.415 | Nightmare. I wake up frightened. I was being chased.
---
0.391 | I was being chased by two people. It was a man and a woman. They chased me into a house where a stranger was sitting at a desk trying to make a computer run on old batteries. I went to find some and w
---
0.371 | It was a dream similar to the one on the 6th of September; the same familiar characters (schoolmates) and others I have never seen or I don't remember. My dream was still centered around the girl I lo
---
0.349 | Another dream I had was that I was being chased by government agents because they had done certain things that I was against.
---
0.325 | I and several other people -- I don't believe I knew any of these people in particular, but they were all about my age ---were being chased by this group of other people, some of them older people, so
---
0.295 | Through The Bowling Alley  I'm at Carter School. I'm leading a group of younger kids. We're being pursued. We get out of the school through the bowlin

Baseline TF-IDF. Lexikalische Suche funktioniert bei wörtlicher Übereinstimmung (bester Treffer 0,413). Beim Synonym „pursued“ statt „chased“ fällt der beste Wert auf 0,293, und die inhaltlich passenden Treffer aus der ersten Anfrage erscheinen gar nicht mehr. Bei deutscher Anfrage versagt das Verfahren vollständig (Höchstwert 0,27, Treffer rein zufällig über Ländernamen). Grund: TF-IDF vergleicht Zeichenketten, keine Bedeutung.